In [1]:
import warnings
warnings.filterwarnings("ignore")

from datetime import datetime
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql import Window
from pyspark.sql.types import StringType
from IPython.display import display

pd.DataFrame.iteritems = pd.DataFrame.items

In [2]:
spark = SparkSession.builder\
       .master("local[*]")\
       .appName("SparklingWaterApp")\
       .config("spark.executor.memory", "6g")\
       .config("spark.driver.memory", "4g")\
       .config("spark.driver.extraJavaOptions",
                "--add-opens java.base/sun.net.www.protocol.http=ALL-UNNAMED "
                "--add-opens java.base/sun.net.www.protocol.https=ALL-UNNAMED "
                "--add-opens java.base/sun.net.www.protocol.jar=ALL-UNNAMED")\
       .config("spark.executor.extraJavaOptions",
                "--add-opens java.base/sun.net.www.protocol.http=ALL-UNNAMED "
                "--add-opens java.base/sun.net.www.protocol.https=ALL-UNNAMED "
                "--add-opens java.base/sun.net.www.protocol.jar=ALL-UNNAMED")\
       .getOrCreate()

In [3]:
spark.sparkContext.setCheckpointDir("/tmp/checkpoints")

In [4]:
spark.sparkContext

<SparkContext master=local[*] appName=SparklingWaterApp>

In [5]:
# pip install h2o_pysparkling_3.5

In [6]:
from pysparkling import *
import h2o
conf = H2OConf().setLogLevel("ERROR") # WARN
hc = H2OContext.getOrCreate(conf)

Connecting to H2O server at http://a0cce2f5eb87:54323 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,08 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.6
H2O_cluster_version_age:,"1 year, 4 months and 22 days"
H2O_cluster_name:,sparkling-water-jovyan_local-1774359386856
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,4 Gb
H2O_cluster_total_cores:,4
H2O_cluster_allowed_cores:,4
H2O_cluster_status:,"locked, healthy"



Sparkling Water Context:
 * Sparkling Water Version: 3.46.0.6-1-3.5
 * H2O name: sparkling-water-jovyan_local-1774359386856
 * cluster size: 1
 * list of used nodes:
  (executorId, host, port)
  ------------------------
  (0,172.17.0.2,54321)
  ------------------------

  Open H2O Flow in browser: http://a0cce2f5eb87:54323 (CMD + click in Mac OSX)

    


In [7]:
frame = h2o.import_file("loan.csv")
df = hc.asSparkFrame(frame)

window = Window.orderBy(f.lit('A'))
df = df.select(f.row_number().over(window).alias("id"), "*")

display(df.limit(5).toPandas())

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


,id,loan_amnt,term,int_rate,emp_length,home_ownership,annual_inc,purpose,addr_state,dti,delinq_2yrs,revol_util,total_acc,bad_loan,longest_credit_length,verification_status
0,1,5000,36 months,10.65,10,RENT,24000.0,credit_card,AZ,27.65,0,83.7,9,0,26,verified
1,2,2500,60 months,15.27,0,RENT,30000.0,car,GA,1.00,0,9.4,4,1,12,verified
2,3,2400,36 months,15.96,10,RENT,12252.0,small_business,IL,8.72,0,98.5,10,0,10,not verified
3,4,10000,36 months,13.49,10,RENT,49200.0,other,CA,20.00,0,21.0,37,0,15,verified
4,5,5000,36 months,7.90,3,RENT,36000.0,wedding,AZ,11.20,0,28.3,12,0,7,verified


In [8]:
# coonstant or id columns to drop

target = "bad_loan"

cols_to_ignore =  ["id", "addr_state", target]

init_features_list = [colu for colu in df.columns if colu.lower() not in cols_to_ignore]

agg_tab = df.agg(f.count(target).alias('count'),
            f.sum(target).alias(f'sum_{target}'),
            f.mean(target).alias(f'mean_{target}'),
            ).toPandas()

display(agg_tab)

,count,sum_bad_loan,mean_bad_loan
0,999,194,0.194194


In [9]:
# Splitting the DataFrame into training and testing sets with stratified sampling
strata = target

strata_list = df.select(strata).distinct().orderBy(strata).rdd.flatMap(lambda x: x).collect()
stratified_dfs = []

train_df = df.limit(0)
test_df = df.limit(0)

for s in strata_list:
    stratified_df = df.filter(df[strata] == s)
    train_strata, test_strata = stratified_df.randomSplit([0.7, 0.3], seed = 42)
    train_df = train_df.union(train_strata)
    test_df = test_df.union(test_strata)

In [10]:
agg_tab1 = train_df.agg(f.count(target).alias('count'),
            f.sum(target).alias(f'sum_{target}'),
            f.mean(target).alias(f'mean_{target}'),
            ).toPandas()

agg_tab1['dataset'] = 'train'

agg_tab2 = test_df.agg(f.count(target).alias('count'),
            f.sum(target).alias(f'sum_{target}'),
            f.mean(target).alias(f'mean_{target}'),
            ).toPandas()

agg_tab2['dataset'] = 'test'

agg_tab = pd.concat([agg_tab1, agg_tab2], ignore_index=True)
cols = ['dataset'] + [col for col in agg_tab.columns if col != 'dataset']

display(agg_tab[cols])

,dataset,count,sum_bad_loan,mean_bad_loan
0,train,735,139,0.189116
1,test,264,55,0.208333


In [11]:
string_cols = [x.name for x in train_df.select(init_features_list).schema.fields if isinstance(x.dataType, StringType)]
display(train_df.select(string_cols).limit(5).toPandas())

numeric_cols = [x.name for x in train_df.select(init_features_list).schema.fields if not isinstance(x.dataType, StringType)]
display(train_df.select(numeric_cols).limit(5).toPandas())

,term,home_ownership,purpose,verification_status
0,36 months,RENT,credit_card,verified
1,36 months,RENT,small_business,not verified
2,36 months,RENT,wedding,verified
3,36 months,RENT,car,verified
4,60 months,OWN,debt_consolidation,not verified


,loan_amnt,int_rate,emp_length,annual_inc,dti,delinq_2yrs,revol_util,total_acc,longest_credit_length
0,5000,10.65,10,24000.0,27.65,0,83.7,9,26
1,2400,15.96,10,12252.0,8.72,0,98.5,10,10
2,5000,7.90,3,36000.0,11.20,0,28.3,12,7
3,3000,18.64,9,48000.0,5.35,0,87.5,4,4
4,6500,14.65,5,72000.0,16.12,0,20.6,23,13


In [12]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as f
from pyspark.ml.feature import QuantileDiscretizer, Bucketizer

def fine_classing(
    df: DataFrame,
    col_name: str,
    n_bins: int = 10,
    relative_error: float = 0.001,
    handle_invalid: str = "keep",  # "keep", "skip", "error"
) -> DataFrame:
    """
    Perform fine classing (binning) for one numeric column in Spark DataFrame.

    Parameters
    ----------
    df : Spark DataFrame
    col_name : str
        Numeric input column to bin.
    n_bins : int
        Number of fine bins.
    relative_error : float
        Used in approx quantiles for equal_width boundaries.
    handle_invalid : str
        How to handle null/NaN in Spark transformers.

    Returns
    -------
    Return only fitted QuantileDiscretizerModel (no transform).
    """
    output_col = f"{col_name}_fine_bin"

    discretizer = QuantileDiscretizer(
        numBuckets=n_bins,
        inputCol=col_name,
        outputCol=output_col,
        handleInvalid=handle_invalid,
        relativeError=relative_error,
    )

    model = discretizer.fit(df.select(col_name))

    return model


def apply_fine_classing_model(model, df: DataFrame) -> DataFrame:
    """
    Apply a fitted QuantileDiscretizerModel (or similar Spark ML model)
    on a Spark DataFrame and return transformed DataFrame.
    """
    df_transformed = model.transform(df)
    output_col = model.getOutputCol()

    df_transformed = (
        df_transformed.withColumn(output_col, f.col(output_col) + f.lit(1))
        .fillna({output_col: 0})
        .withColumn(output_col, f.col(output_col).cast("int"))
    )

    return df_transformed

In [13]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as f

def woe_categorical(
    df: DataFrame,
    feature_col: str,
    target_col: str,
    event_value: int = 1,
    eps: float = 0.5
) -> DataFrame:
    """
    WoE for a categorical feature column in a Spark DataFrame.

    Returns a DataFrame with:
    - category, total, events, non_events
    - dist_event, dist_non_event
    - woe, iv_component
    """
    if event_value is None:
        event_value = (
            df.agg(f.max(f.col(target_col)).alias("event_value"))
            .first()["event_value"]
        )

    # Pre-process target and feature casting
    base_df = df.select(
        f.col(feature_col).alias("category"),
        f.when(f.col(target_col) == f.lit(event_value), 1)
         .otherwise(0)
         .alias("is_event")
    )

    # Aggregate counts per category
    counts_df = (
        base_df.groupBy("category")
        .agg(
            f.count(f.lit(1)).alias("total"),
            f.sum("is_event").alias("events")
        )
        .withColumn("non_events", f.col("total") - f.col("events"))
    )

    # Calculate global totals for distributions
    totals = counts_df.select(
        f.sum("events").alias("total_events"),
        f.sum("non_events").alias("total_non_events")
    ).first()

    total_ev = totals["total_events"]
    total_nev = totals["total_non_events"]

    # Compute WoE and IV components
    result = (
        counts_df
        .withColumn("dist_event", (f.col("events") + eps) / (total_ev + eps))
        .withColumn("dist_non_event", (f.col("non_events") + eps) / (total_nev + eps))
        .withColumn("woe", f.log(f.col("dist_non_event") / f.col("dist_event")))
        .withColumn(
            "iv_component", 
            (f.col("dist_non_event") - f.col("dist_event")) * f.col("woe")
        )
        .select(
            f.col("category").alias(feature_col),
            "total", "events", "non_events",
            f.round("dist_event", 6).alias("dist_event"),
            f.round("dist_non_event", 6).alias("dist_non_event"),
            f.round("woe", 6).alias("woe"),
            f.round("iv_component", 6).alias("iv_component")
        )
        .orderBy("woe")
    )

    return result

In [14]:
train_df_woe = train_df
test_df_woe = test_df

final_features_list = []

for col_name in init_features_list:
    if col_name in numeric_cols:
        print(f"{col_name} -> ", end="")
        
        model = fine_classing(train_df_woe, col_name, 10)
        
        train_df_woe = apply_fine_classing_model(model, train_df_woe)
        train_df_woe = train_df_woe.drop(col_name)

        test_df_woe = apply_fine_classing_model(model, test_df_woe)
        test_df_woe = test_df_woe.drop(col_name)

        output_col_new = f"{col_name}_fine_bin"
        col_name_woe = f"{output_col_new}_woe"

        df_woe = woe_categorical(train_df_woe.select(output_col_new, target), output_col_new, target)\
        .select(output_col_new, f.col("woe").alias(col_name_woe))
        
        train_df_woe = train_df_woe.join(df_woe, on=output_col_new)
        train_df_woe = train_df_woe.drop(output_col_new)
        
        test_df_woe = test_df_woe.join(df_woe, on=output_col_new)
        test_df_woe = test_df_woe.drop(output_col_new)

        final_features_list.append(col_name_woe)
        print(col_name_woe)
        
    if col_name in string_cols:
        print(f"{col_name} -> ", end="")
        
        missing_label = "__MISSING__"
        train_df_woe = train_df_woe.fillna({col_name: missing_label})
        test_df_woe = test_df_woe.fillna({col_name: missing_label})
        
        col_name_woe = f"{col_name}_woe"
        df_woe = woe_categorical(train_df_woe.select(col_name, target), col_name, target)\
        .select(col_name, f.col("woe").alias(col_name_woe))

        train_df_woe = train_df_woe.join(df_woe, on=col_name)
        train_df_woe = train_df_woe.drop(col_name)
        
        test_df_woe = test_df_woe.join(df_woe, on=col_name)
        test_df_woe = test_df_woe.drop(col_name)

        final_features_list.append(col_name_woe)
        print(col_name_woe)

    train_df_woe = train_df_woe.checkpoint()
    test_df_woe = test_df_woe.checkpoint()
        
keep_cols = final_features_list + [target]

train_df_woe = train_df_woe.select(keep_cols).checkpoint()
test_df_woe = test_df_woe.select(keep_cols).checkpoint()

loan_amnt -> loan_amnt_fine_bin_woe
term -> term_woe
int_rate -> int_rate_fine_bin_woe
emp_length -> emp_length_fine_bin_woe
home_ownership -> home_ownership_woe
annual_inc -> annual_inc_fine_bin_woe
purpose -> purpose_woe
dti -> dti_fine_bin_woe
delinq_2yrs -> delinq_2yrs_fine_bin_woe
revol_util -> revol_util_fine_bin_woe
total_acc -> total_acc_fine_bin_woe
longest_credit_length -> longest_credit_length_fine_bin_woe
verification_status -> verification_status_woe


In [15]:
train_h2o =  hc.asH2OFrame(train_df)
test_h2o =  hc.asH2OFrame(test_df)

train_h2o_woe =  hc.asH2OFrame(train_df_woe)
test_h2o_woe =  hc.asH2OFrame(test_df_woe)

In [16]:
from h2o.estimators import H2OModelSelectionEstimator

h2o.display.toggle_user_tips(False)

predictors = final_features_list

model_bw = H2OModelSelectionEstimator(
    family = "binomial",
    mode = "backward",
    min_predictor_number=2,      # stop when 2 predictors remain
    p_values_threshold=0.05,     # OR stop when all p-values <= 0.05
    compute_p_values=True,       # required for p_values_threshold to work
    seed=42
)

model_bw.train(x=predictors, y=target, training_frame=train_h2o_woe)

modelselection Model Build progress: |███████████████████████████████████████████| (done) 100%


,coefficient_names,predictor_names,z_values,p_values,predictors_removed
with 9 predictors,"loan_amnt_fine_bin_woe, term_woe, int_rate_fine_bin_woe, emp_length_fine_bin_woe, annual_inc_fine_bin_woe, purpose_woe, dti_fine_bin_woe, revol_util_fine_bin_woe, longest_credit_length_fine_bin_woe, Intercept","loan_amnt_fine_bin_woe, term_woe, int_rate_fine_bin_woe, emp_length_fine_bin_woe, annual_inc_fine_bin_woe, purpose_woe, dti_fine_bin_woe, revol_util_fine_bin_woe, longest_credit_length_fine_bin_woe","-2.0396103697214643, -2.7941933591837778, -3.6991141695777117, -2.858370240736907, -2.5007300016431793, -4.431951731513949, -3.890251672592889, -2.2008808450498236, -2.1892710466398513, -13.9036302564158","0.04138914840224216, 0.005202935853360649, 2.1635326807328096E-4, 0.004258232268947433, 0.012393762615481214, 9.338392568771608E-6, 1.0014030904951367E-4, 0.027744460310449137, 0.028577145459667434, 6.020846415625877E-44",loan_amnt_fine_bin_woe
with 10 predictors,"loan_amnt_fine_bin_woe, term_woe, int_rate_fine_bin_woe, emp_length_fine_bin_woe, annual_inc_fine_bin_woe, purpose_woe, dti_fine_bin_woe, revol_util_fine_bin_woe, longest_credit_length_fine_bin_woe, verification_status_woe, Intercept","loan_amnt_fine_bin_woe, term_woe, int_rate_fine_bin_woe, emp_length_fine_bin_woe, annual_inc_fine_bin_woe, purpose_woe, dti_fine_bin_woe, revol_util_fine_bin_woe, longest_credit_length_fine_bin_woe, verification_status_woe","-2.2518471429777613, -3.0082941161009153, -3.836584919616944, -2.970706672949621, -2.4082110677766546, -4.4084120762746455, -3.8533930395619764, -2.2797070560708157, -2.193875072366717, 1.8300219890683629, -13.792291383520334","0.024331933350164293, 0.002627187452884345, 1.2475706945545704E-4, 0.0029711541409505942, 0.016030909924183635, 1.0413127773324195E-5, 1.1649216519321903E-4, 0.02262506854159611, 0.028244391909501957, 0.06724665082450199, 2.8360648971802137E-43",verification_status_woe
with 11 predictors,"loan_amnt_fine_bin_woe, term_woe, int_rate_fine_bin_woe, emp_length_fine_bin_woe, home_ownership_woe, annual_inc_fine_bin_woe, purpose_woe, dti_fine_bin_woe, revol_util_fine_bin_woe, longest_credit_length_fine_bin_woe, verification_status_woe, Intercept","loan_amnt_fine_bin_woe, term_woe, int_rate_fine_bin_woe, emp_length_fine_bin_woe, home_ownership_woe, annual_inc_fine_bin_woe, purpose_woe, dti_fine_bin_woe, revol_util_fine_bin_woe, longest_credit_length_fine_bin_woe, verification_status_woe","-2.301923527011381, -3.0817158964684523, -3.8082471719741178, -3.073029348681677, -1.5058718303052872, -2.175299969532754, -4.542951734636782, -3.83007343097706, -2.181436314467377, -2.17014765171156, 1.8174597765522784, -13.783859995232639","0.02133948517096853, 0.0020581116675434706, 1.3995534271214876E-4, 0.0021189764684035406, 0.13210007924883194, 0.02960764481767935, 5.54719559216414E-6, 1.281050252066293E-4, 0.029151160110727773, 0.02999566227782891, 0.06914674193735008, 3.1876224966053633E-43",home_ownership_woe
with 12 predictors,"loan_amnt_fine_bin_woe, term_woe, int_rate_fine_bin_woe, emp_length_fine_bin_woe, home_ownership_woe, annual_inc_fine_bin_woe, purpose_woe, dti_fine_bin_woe, revol_util_fine_bin_woe, total_acc_fine_bin_woe, longest_credit_length_fine_bin_woe, verification_status_woe, Intercept","loan_amnt_fine_bin_woe, term_woe, int_rate_fine_bin_woe, emp_length_fine_bin_woe, home_ownership_woe, annual_inc_fine_bin_woe, purpose_woe, dti_fine_bin_woe, revol_util_fine_bin_woe, total_acc_fine_bin_woe, longest_credit_length_fine_bin_woe, verification_status_woe","-2.292523587810121, -3.1597262902765624, -3.692235051149964, -3.017642784437041, -1.3677363293570033, -2.1292097022501335, -4.341870256201415, -3.8545190696051366, -2.217665485929763, -1.3369748990508965, -2.1160474194821663, 1.8231791610704708, -13.793918084612429","0.021875448025973982, 0.0015791741547604493, 2.222918714271467E-4, 0.002547489985328701, 0.1713946239267391, 0.0332369145468732, 1.412749551240072E-5, 1.1595734558318558E-4, 0.026577649

In [17]:
pred_subsets = model_bw.get_best_model_predictors()

# Inspect all subsets
for i, subset in enumerate(pred_subsets):
    print(f"Step {i} ({len(subset)} predictors): {subset}")

final_predictors = pred_subsets[0]
full_predictors = pred_subsets[-1]

Step 0 (9 predictors): ['loan_amnt_fine_bin_woe', 'term_woe', 'int_rate_fine_bin_woe', 'emp_length_fine_bin_woe', 'annual_inc_fine_bin_woe', 'purpose_woe', 'dti_fine_bin_woe', 'revol_util_fine_bin_woe', 'longest_credit_length_fine_bin_woe']
Step 1 (10 predictors): ['loan_amnt_fine_bin_woe', 'term_woe', 'int_rate_fine_bin_woe', 'emp_length_fine_bin_woe', 'annual_inc_fine_bin_woe', 'purpose_woe', 'dti_fine_bin_woe', 'revol_util_fine_bin_woe', 'longest_credit_length_fine_bin_woe', 'verification_status_woe']
Step 2 (11 predictors): ['loan_amnt_fine_bin_woe', 'term_woe', 'int_rate_fine_bin_woe', 'emp_length_fine_bin_woe', 'home_ownership_woe', 'annual_inc_fine_bin_woe', 'purpose_woe', 'dti_fine_bin_woe', 'revol_util_fine_bin_woe', 'longest_credit_length_fine_bin_woe', 'verification_status_woe']
Step 3 (12 predictors): ['loan_amnt_fine_bin_woe', 'term_woe', 'int_rate_fine_bin_woe', 'emp_length_fine_bin_woe', 'home_ownership_woe', 'annual_inc_fine_bin_woe', 'purpose_woe', 'dti_fine_bin_woe', 

In [18]:
from pysparkling.ml import H2OGLMClassifier

predictors = final_predictors

glm_estimator = H2OGLMClassifier(
        family = "binomial",
        featuresCols = predictors,
        labelCol = target,
        calcLike = True,
        lambdaValue = [0],
        lambdaSearch = False,
        computePValues = True,
        solver = "IRLSM",
        link = "logit",
        )

model = glm_estimator.fit(train_df_woe)
print(model.getModelSummary)

gini = model.getTrainingMetrics()['Gini']
print(f"Gini: {round(gini, 6)}")

display(model.getCoefficients().withColumn("p value", f.round("p value", 6)).toPandas())

<bound method H2OMOJOModelParams.getModelSummary of Model Details
H2OGLM
Model Key: GLM_92b666938bba

Model summary
Family: binomial
Link: logit
Regularization: None
Number of Predictors Total: 9
Number of Active Predictors: 9
Number of Iterations: 5
Training Frame: frame_rdd_638771052469

Training metrics
PRAUC: 0.46451013011649056
Nobs: 735.0
Logloss: 0.393800136207194
Gini: 0.5972067983197333
RMSE: 0.3557464889651662
ResidualDeviance: 578.8862002245752
NullDeviance: 712.8590052906397
ScoringTime: 1.774359425384E12
Loglikelihood: -289.4431001122876
MSE: 0.1265555644110431
R2: 0.17473227048481788
NullDegreesOfFreedom: 734.0
MeanPerClassError: 0.2776423156776592
AUC: 0.7986033991598667
AIC: -558.8862002245752
ResidualDegreesOfFreedom: 725.0

More info available using methods like:
getFeatureImportances(), getScoringHistory(), getCrossValidationScoringHistory()>
Gini: 0.597207


,names,Coefficients,Std. Error,z value,p value,Standardized Coefficients
0,Intercept,-1.656482,0.119140,-13.903630,0.000000,-1.903583
1,loan_amnt_fine_bin_woe,-0.894428,0.438529,-2.039610,0.041389,-0.245157
2,term_woe,-0.585300,0.209470,-2.794193,0.005203,-0.289862
3,int_rate_fine_bin_woe,-0.662558,0.179113,-3.699114,0.000216,-0.638916
4,emp_length_fine_bin_woe,-1.072159,0.375094,-2.858370,0.004258,-0.296508
5,annual_inc_fine_bin_woe,-0.875405,0.350060,-2.500730,0.012394,-0.289096
6,purpose_woe,-1.372515,0.309686,-4.431952,0.000009,-0.496637
7,dti_fine_bin_woe,-1.033579,0.265684,-3.890252,0.000100,-0.458937
8,revol_util_fine_bin_woe,-0.641536,0.291490,-2.200881,0.027744,-0.254237
9,longest_credit_length_fine_bin_woe,-1.006617,0.459796,-2.189271,0.028577,-0.244388


In [19]:
display(model.getFeatureImportances().toPandas())

,Variable,Relative Importance,Scaled Importance,Percentage
0,int_rate_fine_bin_woe,0.638916,1.000000,0.198808
1,purpose_woe,0.496637,0.777312,0.154536
2,dti_fine_bin_woe,0.458937,0.718305,0.142805
3,emp_length_fine_bin_woe,0.296508,0.464080,0.092263
4,term_woe,0.289862,0.453677,0.090195
5,annual_inc_fine_bin_woe,0.289096,0.452479,0.089956
6,revol_util_fine_bin_woe,0.254237,0.397920,0.079110
7,loan_amnt_fine_bin_woe,0.245157,0.383707,0.076284
8,longest_credit_length_fine_bin_woe,0.244388,0.382504,0.076045


In [20]:
display(model.transform(test_df_woe).crosstab(target, "prediction").toPandas())

,bad_loan_prediction,0,1
0,0,144,65
1,1,25,30


In [21]:
from pysparkling.ml import H2OXGBoostClassifier

min_rows = int(train_df.count() * 0.05)

xgb_classifier = H2OXGBoostClassifier(labelCol = target, 
                                 booster = "gbtree",
                                 ntrees = 250, 
                                 minRows = min_rows,
                                 detailedPredictionCol = "prediction")

keep_cols = numeric_cols + string_cols + [target]
model = xgb_classifier.fit(train_df.select(keep_cols))
print(model.getModelSummary)

<bound method H2OMOJOModelParams.getModelSummary of Model Details
H2OXGBoost
Model Key: XGBoost_8232048c69f6

Model summary
Number of Trees: 250

Training metrics
PRAUC: 0.404733462211068
Nobs: 735.0
Logloss: 0.4241422787109303
Gini: 0.5145454106513447
RMSE: 0.36602782468888795
ScoringTime: 1.774359432249E12
Loglikelihood: NaN
MSE: 0.13397636844647928
R2: 0.12634127222249947
MeanPerClassError: 0.295615856308242
AUC: 0.7572727053256724
AIC: NaN

More info available using methods like:
getFeatureImportances(), getScoringHistory(), getCrossValidationScoringHistory()>


In [22]:
display(model.getFeatureImportances().toPandas())

,Variable,Relative Importance,Scaled Importance,Percentage
0,int_rate,70.024689,1.000000,0.439647
1,dti,19.789263,0.282604,0.124246
2,revol_util,16.188408,0.231181,0.101638
3,annual_inc,12.767155,0.182324,0.080158
4,loan_amnt,11.429572,0.163222,0.071760
5,total_acc,10.467224,0.149479,0.065718
6,emp_length,6.949555,0.099244,0.043633
7,purpose.credit_card,5.196002,0.074202,0.032623
8,longest_credit_length,3.361048,0.047998,0.021102
9,verification_status.not verified,1.889930,0.026989,0.011866


In [23]:
display(model.transform(test_df).crosstab(target, "prediction").toPandas())

,bad_loan_prediction,0,1
0,0,132,77
1,1,26,29


In [24]:
from pysparkling.ml import H2ODeepLearning

all_features = numeric_cols + string_cols

estimator = H2ODeepLearning(
                distribution = "bernoulli",
                hidden = [100, 50, 20],
                epochs = 1000,
                columnsToCategorical = string_cols,
                featuresCols = all_features,
                labelCol = target,
                detailedPredictionCol = "prediction")

keep_cols = all_features + [target]
model = estimator.fit(train_df.select(keep_cols))
print(model.getModelSummary)

<bound method H2OMOJOModelParams.getModelSummary of Model Details
H2ODeepLearning
Model Key: DeepLearning_3bce78fb559c

Model summary
Layer: 1
Units: 33
Type: Input
Dropout: 0.0
L1: null
L2: null
Mean Rate: null
Rate RMS: null
Momentum: null
Mean Weight: null
Weight RMS: null
Mean Bias: null
Bias RMS: null

Layer: 2
Units: 100
Type: Rectifier
Dropout: 0.0
L1: 0.0
L2: 0.0
Mean Rate: 0.4013985758565435
Rate RMS: 0.4201698303222656
Momentum: 0.0
Mean Weight: -0.006523855178101891
Weight RMS: 0.15011996030807495
Mean Bias: 0.4361993847139602
Bias RMS: 0.09311684966087341

Layer: 3
Units: 50
Type: Rectifier
Dropout: 0.0
L1: 0.0
L2: 0.0
Mean Rate: 0.6288889659038279
Rate RMS: 0.37159812450408936
Momentum: 0.0
Mean Weight: -0.007193101304748052
Weight RMS: 0.1679636836051941
Mean Bias: 0.9761548906553653
Bias RMS: 0.06659209728240967

Layer: 4
Units: 20
Type: Rectifier
Dropout: 0.0
L1: 0.0
L2: 0.0
Mean Rate: 0.3800070230240235
Rate RMS: 0.37946057319641113
Momentum: 0.0
Mean Weight: -0.016862

In [25]:
display(model.getFeatureImportances().toPandas())

,Variable,Relative Importance,Scaled Importance,Percentage
0,longest_credit_length,1.000000,1.000000,0.042865
1,annual_inc,0.984520,0.984520,0.042201
2,total_acc,0.974792,0.974792,0.041784
3,revol_util,0.973983,0.973983,0.041749
4,int_rate,0.959858,0.959858,0.041144
5,emp_length,0.948500,0.948500,0.040657
6,loan_amnt,0.867867,0.867867,0.037201
7,purpose.credit_card,0.864083,0.864083,0.037039
8,purpose.other,0.849947,0.849947,0.036433
9,dti,0.836139,0.836139,0.035841


In [26]:
display(model.transform(test_df).crosstab(target, "prediction").toPandas())

,bad_loan_prediction,0,1
0,0,182,27
1,1,46,9
